#### Arithmetic Expressions with Constants in P0 (WASM)

In [ ]:
import import_ipynb
from P0 import compileString

Analyze the generated code of the following program:

In [ ]:
print(
  compileString("""
var x: integer
program p
  var y: integer
    y := x + 3 + 4
    y := 3 + 4 + x
    y := 3 + x + 4
""")
)

The second assignment adds `7`, but the first and third add `3` and `4` separately. Explain briefly why the P0 compiler generates that code!

The P0 parser evaluates expressions left-to-right following the grammar's associativity. In `simpleExpression()`, the `while` loop processes additive operators left-to-right. Constant folding in P0 only triggers when BOTH operands of a binary operation are `Const` (the check `type(x) == Const == type(y)`).

- `y := x + 3 + 4`: parsed as `(x + 3) + 4`. The first operand `x` is `Var`, `3` is `Const`. Since they're not both `Const`, `CG.genBinaryOp(PLUS, x, 3)` emits `i32.add`. The result is a stack `Var`, so `(result) + 4` also emits `i32.add`. Two additions generated.

- `y := 3 + 4 + x`: parsed as `(3 + 4) + x`. Both `3` and `4` are `Const`, so constant folding fires: `x.val = 3 + 4 = 7`. Then `7 + x` where `7` is `Const` and `x` is `Var`, not both `Const`, so one `genBinaryOp` call. One addition generated with `i32.const 7`.

- `y := 3 + x + 4`: parsed as `(3 + x) + 4`. `3` is `Const`, `x` is `Var`, not both `Const`, so `genBinaryOp` emits `i32.add`. Then `(result) + 4` also emits `i32.add`. Two additions generated.

Constant folding is a peephole optimization that only applies when both operands are compile-time constants. Left-to-right evaluation means only `3 + 4 + x` gets the two constants adjacent in the parse tree.

With a restricted range of integers, some programming languages implement strict arithmetic, meaning that overflow is an error, and some languages implement modulo arithmetic, meaning that there is no overflow, but the result "wraps around". Are compilers allowed to optimize the first and third assignments to add `7` with strict arithmetic and modulo arithmetic?

**Strict arithmetic** (overflow is an error): yes. Since both constants 3 and 4 are positive, the overflow behavior is preserved. If `x + 3` overflows (positive), then `x + 7` also overflows since 7 > 3. If `x + 3` does not overflow but `(x + 3) + 4` does, then `x + 7` also overflows since the mathematical sum is the same: `x + 3 + 4 = x + 7`. The optimization would NOT be valid in general if the constants had different signs (e.g. `x + MAX_INT + (-1)` could overflow at the first step while `x + (MAX_INT - 1)` might not).

**Modulo arithmetic** (wraparound): yes. Two's complement addition is associative: `(x + 3) + 4 ≡ x + (3 + 4) ≡ x + 7 (mod 2^n)` for any x. The optimization is always valid with modulo arithmetic regardless of the constants' signs, because modular addition forms an abelian group.